# Settings

In [2]:
from pathlib import Path

import vine_data as vd

H5_PATH = vd.DEFAULT_H5_PATH

# Load Labels

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Open read-only; keep the handle for later cells
f = vd.open_h5(H5_PATH)
wavelengths = vd.load_wavelengths(f)

print("\nroot attrs:")
for k, v in f.attrs.items():
    print(f"  {k}: {v}")
print(f"\nwavelengths: {wavelengths.shape[0]} bands, "
      f"{wavelengths.min():.1f}-{wavelengths.max():.1f} nm")

meta = vd.index_vines(f)
print(f"{len(meta)} vines indexed")
display(meta.head(5))


root attrs:
  class_names: ['Negative' 'Positive']
  label_names: ['red_blotch' 'leafroll3']
  label_year: 2026
  layout: cube is (lines, bands, samples) as stored in the ENVI BIL files
  sensor: Specim IQ
  sensor_docs: docs/Specim-IQ-Datasheet.pdf

wavelengths: 204 bands, 400.0-1000.0 nm
998 vines indexed


,path,variety,row,vine,ccovi_code,capture_date,red_blotch,leafroll3,label,cube_shape,dtype
0,Baco/R1/V1,Baco,R1,1,B01,2026-08-12,1,1,"[1, 1]","(151, 204, 157)",float32
1,Baco/R1/V2,Baco,R1,2,B02,2026-08-12,1,1,"[1, 1]","(158, 204, 151)",float32
2,Baco/R1/V3,Baco,R1,3,B03,2026-08-12,1,0,"[1, 0]","(141, 204, 150)",float32
3,Baco/R1/V4,Baco,R1,4,B04,2026-08-12,1,0,"[1, 0]","(158, 204, 153)",float32
4,Baco/R1/V5,Baco,R1,5,B05,2026-08-12,0,1,"[0, 1]","(140, 204, 171)",float32


In [ ]:
# First vine's cube lives at f[path]["cube"] — (lines, bands, samples) = (height, 204, width).
# Values are reflectance (float32), typically 0–1; >1.0 is a specular glint.
row = meta.iloc[0]
print(row[["path", "variety", "cube_shape", "red_blotch", "leafroll3"]])

cube = f[row["path"]]["cube"]
print(f"\ndataset: {cube}")
print(f"shape {cube.shape}  dtype {cube.dtype}")
print(f"min {cube[:].min():.4f}  mean {cube[:].mean():.4f}  max {cube[:].max():.4f}")

h, n_bands, w = cube.shape
y, x = h // 2, w // 2
spectrum = cube[y, :, x]  # one pixel, all 204 bands
print(f"\ncentre pixel (line={y}, sample={x}):")
print(f"  min/mean/max {spectrum.min():.4f} / {spectrum.mean():.4f} / {spectrum.max():.4f}")
print("  first 12 bands:")
for nm, v in zip(wavelengths[:12], spectrum[:12]):
    print(f"    {nm:7.1f} nm   {v:.6f}")

band_i = int(np.argmin(np.abs(wavelengths - 550)))
patch = np.asarray(cube[y - 2:y + 3, band_i, x - 2:x + 3]) 
print(f"\n5x5 patch at {wavelengths[band_i]:.1f} nm (band {band_i}):")
np.set_printoptions(precision=4, suppress=True)
print(patch)

path               Baco/R1/V1
variety                  Baco
cube_shape    (151, 204, 157)
red_blotch                  1
leafroll3                   1
Name: 0, dtype: object

dataset: <HDF5 dataset "cube": shape (151, 204, 157), type "<f4">
shape (151, 204, 157)  dtype float32
min 0.0000  mean 0.3632  max 1.3488

centre pixel (line=75, sample=78):
  min/mean/max 0.1243 / 0.6311 / 1.1455
  first 12 bands:
      400.0 nm   0.730769
      403.0 nm   0.606061
      405.9 nm   0.538462
      408.9 nm   0.446809
      411.8 nm   0.403509
      414.8 nm   0.348485
      417.7 nm   0.333333
      420.7 nm   0.301205
      423.6 nm   0.282609
      426.6 nm   0.287129
      429.6 nm   0.256637
      432.5 nm   0.256000

5x5 patch at 550.7 nm (band 51):
[[0.2885 0.2863 0.2797 0.2654 0.2896]
 [0.2933 0.2889 0.2811 0.2657 0.2778]
 [0.3161 0.3062 0.2952 0.2731 0.2742]
 [0.3007 0.304  0.3084 0.2852 0.2764]
 [0.2775 0.2775 0.2775 0.2698 0.2709]]


In [ ]:
def fmt(count, n):
    return f"{int(count)} ({100 * count / n:.1f}%)".replace(".0%", "%")

# Mutually exclusive status: RB only / LR only / both / healthy
rows_status = []
for variety, g in meta.groupby("variety", sort=False):
    n = len(g)
    rb_only = ((g.red_blotch == 1) & (g.leafroll3 == 0)).sum()
    lr_only = ((g.red_blotch == 0) & (g.leafroll3 == 1)).sum()
    both = ((g.red_blotch == 1) & (g.leafroll3 == 1)).sum()
    healthy = ((g.red_blotch == 0) & (g.leafroll3 == 0)).sum()
    rows_status.append({
        "Red Blotch": fmt(rb_only, n),
        "Leafroll": fmt(lr_only, n),
        "Both": fmt(both, n),
        "Healthy": fmt(healthy, n),
    })

status_distribution = pd.DataFrame(
    rows_status, index=list(meta.groupby("variety", sort=False).groups.keys())
)
status_distribution

,Red Blotch,Leafroll,Both,Healthy
Baco,82 (33.1%),54 (21.8%),53 (21.4%),59 (23.8%)
Cab Franc,240 (96%),0 (0%),1 (0.4%),9 (3.6%)
Chardonnay,38 (15.2%),72 (28.8%),50 (20%),90 (36%)
Vidal,157 (62.8%),13 (5.2%),12 (4.8%),68 (27.2%)
